**Data cleaning and preprocessing steps**

In [17]:
import pandas as pd

data = pd.read_csv("TrafficTwoMonth.csv")

print(data["Traffic Situation"].unique())
print(data.head())

['normal' 'low' 'heavy' 'high']
          Time  Date Day of the week  CarCount  BikeCount  BusCount  \
0  12:00:00 AM    10         Tuesday        13          2         2   
1  12:15:00 AM    10         Tuesday        14          1         1   
2  12:30:00 AM    10         Tuesday        10          2         2   
3  12:45:00 AM    10         Tuesday        10          2         2   
4   1:00:00 AM    10         Tuesday        11          2         1   

   TruckCount  Total Traffic Situation  
0          24     41            normal  
1          36     52            normal  
2          32     46            normal  
3          36     50            normal  
4          34     48            normal  


In [18]:
# Convert traffic situation to numerical values
data['Traffic Situation'] = data['Traffic Situation'].map({
    'low': 0,
    'normal': 1,
    'high': 2,
    'heavy': 3
})

print(data["Traffic Situation"].unique())
print(data.head())

[1 0 3 2]
          Time  Date Day of the week  CarCount  BikeCount  BusCount  \
0  12:00:00 AM    10         Tuesday        13          2         2   
1  12:15:00 AM    10         Tuesday        14          1         1   
2  12:30:00 AM    10         Tuesday        10          2         2   
3  12:45:00 AM    10         Tuesday        10          2         2   
4   1:00:00 AM    10         Tuesday        11          2         1   

   TruckCount  Total  Traffic Situation  
0          24     41                  1  
1          36     52                  1  
2          32     46                  1  
3          36     50                  1  
4          34     48                  1  


In [19]:
# Convert time to datetime
data['Time'] = pd.to_datetime(data['Time'])

# Extract useful features
data['Hour'] = data['Time'].dt.hour
data['Minute'] = data['Time'].dt.minute

# Drop original time column
data = data.drop(columns=['Time'])

print(data.head())
print(data.info())

   Date Day of the week  CarCount  BikeCount  BusCount  TruckCount  Total  \
0    10         Tuesday        13          2         2          24     41   
1    10         Tuesday        14          1         1          36     52   
2    10         Tuesday        10          2         2          32     46   
3    10         Tuesday        10          2         2          36     50   
4    10         Tuesday        11          2         1          34     48   

   Traffic Situation  Hour  Minute  
0                  1     0       0  
1                  1     0      15  
2                  1     0      30  
3                  1     0      45  
4                  1     1       0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952 entries, 0 to 5951
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Date               5952 non-null   int64 
 1   Day of the week    5952 non-null   object
 2   CarCount       

/tmp/ipykernel_6871/1436567642.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Time'] = pd.to_datetime(data['Time'])


In [20]:
# Encode day of week to int
data['Day of the week'] = data['Day of the week'].astype('category').cat.codes

print(data.head())
print(data.info())

   Date  Day of the week  CarCount  BikeCount  BusCount  TruckCount  Total  \
0    10                5        13          2         2          24     41   
1    10                5        14          1         1          36     52   
2    10                5        10          2         2          32     46   
3    10                5        10          2         2          36     50   
4    10                5        11          2         1          34     48   

   Traffic Situation  Hour  Minute  
0                  1     0       0  
1                  1     0      15  
2                  1     0      30  
3                  1     0      45  
4                  1     1       0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5952 entries, 0 to 5951
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Date               5952 non-null   int64
 1   Day of the week    5952 non-null   int8 
 2   CarCount     

In [21]:
# Feature selection
features = ['CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Total', 'Hour', 'Minute', 'Day of the week']
target = 'Traffic Situation'

X = data[features]
y = data[target]

print(X)
print(y)

      CarCount  BikeCount  BusCount  TruckCount  Total  Hour  Minute  \
0           13          2         2          24     41     0       0   
1           14          1         1          36     52     0      15   
2           10          2         2          32     46     0      30   
3           10          2         2          36     50     0      45   
4           11          2         1          34     48     1       0   
...        ...        ...       ...         ...    ...   ...     ...   
5947        16          3         1          36     56    22      45   
5948        11          0         1          30     42    23       0   
5949        15          4         1          25     45    23      15   
5950        16          5         0          27     48    23      30   
5951        14          3         1          15     33    23      45   

      Day of the week  
0                   5  
1                   5  
2                   5  
3                   5  
4              

In [27]:
# Normalisation
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled)

[[0.04571429 0.02857143 0.04       ... 0.         0.         0.83333333]
 [0.05142857 0.01428571 0.02       ... 0.         0.33333333 0.83333333]
 [0.02857143 0.02857143 0.04       ... 0.         0.66666667 0.83333333]
 ...
 [0.05714286 0.05714286 0.02       ... 1.         0.33333333 0.66666667]
 [0.06285714 0.07142857 0.         ... 1.         0.66666667 0.66666667]
 [0.05142857 0.04285714 0.02       ... 1.         1.         0.66666667]]


In [28]:
# Create sequences
import numpy as np

sequenceLength = 10  # number of time steps

X_sequences = []
y_sequences = []

for i in range(len(X_scaled) - sequenceLength):
    X_sequences.append(X_scaled[i:i+sequenceLength])
    y_sequences.append(y.iloc[i+sequenceLength])

X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)


print("Shape of X_sequences:")
print(X_sequences.shape)

print("\nShape of y_sequences:")
print(y_sequences.shape)

print("\nExample of one input sequence:")
print(X_sequences[0])

print("\nCorresponding target value:")
print(y_sequences[0])

Shape of X_sequences:
(5942, 10, 8)

Shape of y_sequences:
(5942,)

Example of one input sequence:
[[0.04571429 0.02857143 0.04       0.4        0.07751938 0.
  0.         0.83333333]
 [0.05142857 0.01428571 0.02       0.6        0.12015504 0.
  0.33333333 0.83333333]
 [0.02857143 0.02857143 0.04       0.53333333 0.09689922 0.
  0.66666667 0.83333333]
 [0.02857143 0.02857143 0.04       0.6        0.1124031  0.
  1.         0.83333333]
 [0.03428571 0.02857143 0.02       0.56666667 0.10465116 0.04347826
  0.         0.83333333]
 [0.05714286 0.01428571 0.02       0.65       0.13565891 0.04347826
  0.33333333 0.83333333]
 [0.05142857 0.02857143 0.04       0.45       0.09302326 0.04347826
  0.66666667 0.83333333]
 [0.04571429 0.02857143 0.02       0.33333333 0.05813953 0.04347826
  1.         0.83333333]
 [0.01142857 0.         0.         0.43333333 0.04651163 0.08695652
  0.         0.83333333]
 [0.04571429 0.         0.         0.56666667 0.10077519 0.08695652
  0.33333333 0.83333333]]

C